In [1]:
!pip install -q wandb sentence-transformers scikit-learn

import wandb
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from kaggle_secrets import UserSecretsClient

# Load W&B API key from Kaggle Secrets - no manual login needed
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

run = wandb.init(
    entity="23f2005025-dl-genai-project",
    project="dl-genai-project",
    name="day1-tfidf-baseline",
    job_type="baseline"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: meetbatra (meetbatra-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run ll7mzedo
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260710_084059-ll7mzedo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run day1-tfidf-baseline
wandb: ⭐️ View project at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/r

In [2]:
# Load competition data
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train.head())

# Create a local validation split from train (80/20)
# so we can measure mAP@3 ourselves before submitting
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
print("Train split:", train_split.shape)
print("Val split:", val_split.shape)

Train shape: (2000, 8)
Test shape: (500, 7)
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that 

In [3]:
def apk(actual, predicted, k=3):
    """Average precision at k for a single prediction"""
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
            break  # only one correct answer possible per question
    return score

def mapk(actual, predicted, k=3):
    """Mean average precision at k"""
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])


def tfidf_predict_top3(train_df, val_df, option_cols=['A', 'B', 'C', 'D', 'E']):
    """For each question, rank options by TF-IDF cosine similarity to the prompt"""
    predictions = []

    for idx, row in val_df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        # Fit TF-IDF on prompt + all options for this question
        corpus = [prompt] + options
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(corpus)

        prompt_vec = tfidf_matrix[0:1]
        option_vecs = tfidf_matrix[1:]

        sims = cosine_similarity(prompt_vec, option_vecs)[0]

        # Rank option letters by similarity, descending
        ranked_idx = np.argsort(sims)[::-1]
        ranked_letters = [option_cols[i] for i in ranked_idx]

        predictions.append(ranked_letters[:3])

    return predictions


# Run baseline on validation set
val_predictions = tfidf_predict_top3(train_split, val_split)
val_actual = val_split['answer'].tolist()

score = mapk(val_actual, val_predictions, k=3)
print(f"TF-IDF baseline local mAP@3: {score:.4f}")

wandb.log({"local_map3": score, "approach": "tfidf_baseline"})

TF-IDF baseline local mAP@3: 0.3121


In [4]:
from sentence_transformers import SentenceTransformer

# Load a lightweight but strong sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

def embedding_predict_top3(val_df, model, option_cols=['A', 'B', 'C', 'D', 'E']):
    predictions = []

    for idx, row in val_df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        prompt_emb = model.encode([prompt])
        option_embs = model.encode(options)

        sims = cosine_similarity(prompt_emb, option_embs)[0]

        ranked_idx = np.argsort(sims)[::-1]
        ranked_letters = [option_cols[i] for i in ranked_idx]

        predictions.append(ranked_letters[:3])

    return predictions


val_predictions_emb = embedding_predict_top3(val_split, model)
score_emb = mapk(val_actual, val_predictions_emb, k=3)

print(f"MiniLM embedding local mAP@3: {score_emb:.4f}")

wandb.log({"local_map3": score_emb, "approach": "minilm_embeddings"})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM embedding local mAP@3: 0.3996


In [5]:
# Generate predictions on the actual test set using MiniLM embeddings
test_predictions = embedding_predict_top3(test, model)

# Format submission to match exact sample format: "ID,Prediction"
submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(pred) for pred in test_predictions]
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))

wandb.log({"day": 1, "final_approach": "minilm_embeddings", "local_map3": score_emb})
wandb.finish()

wandb: updating run metadata


   ID Prediction
0   1      B E A
1   2      B D C
2   3      A C D
3   4      A E C
4   5      B C D
5   6      E A D
6   7      E C A
7   8      D A C
8   9      A C E
9  10      C E D


wandb: uploading history steps 2-2, summary, console lines 55-65
wandb: 
wandb: Run history:
wandb:        day ▁
wandb: local_map3 ▁██
wandb: 
wandb: Run summary:
wandb:       approach minilm_embeddings
wandb:            day 1
wandb: final_approach minilm_embeddings
wandb:     local_map3 0.39958
wandb: 
wandb: 🚀 View run day1-tfidf-baseline at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/ll7mzedo
wandb: ⭐️ View project at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260710_084059-ll7mzedo/logs
